# KiCad Standardized Multi-Dataset PCB Pipeline with LabelMe JSON Exporter
### Exporting SAM Polygon Overlays directly for LabelMe GUI

This notebook provides a unified pipeline to download, clean, and process PCB datasets while exporting SAM polygon points directly to **LabelMe JSON Format** (`.json`):
1. **LabelMe JSON Exporter**: Saves SAM polygon points as interactive shapes for LabelMe GUI visual editing.
2. **Label Homogenizer Engine**: Normalizes variations across all datasets into KiCad Standard Classes (`Capacitor_SMD`, `Resistor_SMD`, `Package_SO`, etc.).
3. **Automatic Dataset Downloader**: Pulls WACV, FICS-PCB, PKU-Market, and DeepPCB datasets using `kagglehub`.
4. **Data Inspection & Cleaning**: Filters out invalid bounding boxes, corrupt images, and tiny noise.
5. **Pretrained SAM Point Extractor**: Passes box prompts into Meta's `sam_vit_b.pth` to generate exact `(x, y)` polygon points.
6. **CSV & YOLO Exporter**: Saves polygon points to both **`polygon_points.csv`** and YOLO-seg `.txt` label files.

In [ ]:
# Step 1: Install Dependencies
!pip install torch torchvision opencv-python matplotlib pandas kagglehub git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import cv2
import torch
import random
import json
import pandas as pd
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor
import kagglehub

# Step 2: Define KiCad Classes & Label Homogenizer
KICAD_CLASSES = [
    'Capacitor_SMD',
    'Resistor_SMD',
    'Package_SO',
    'Package_TO_SOT_SMD',
    'Diode_SMD',
    'Connector',
    'Inductor_SMD',
    'Button_Switch_SMD',
    'LED_SMD',
    'Transformer_SMD',
    'PCB_Defect',
    'Unknown_Component'
]

LABEL_SYNONYMS = {
    'Capacitor_SMD': ['c', 'cap', 'capacitor', 'capacitors', 'c_smd', 'c_tht', 'cap1', 'cap2', 'cap3', 'cap4'],
    'Resistor_SMD': ['r', 'res', 'resistor', 'resistors', 'r_smd', 'r_tht'],
    'Package_SO': ['ic', 'chip', 'integrated_circuit', 'soic', 'sop', 'qfp', 'qfn', 'dip', 'mcu'],
    'Package_TO_SOT_SMD': ['q', 'transistor', 'mosfet', 'fet', 'bjt', 'sot', 'sot23', 'to220', 'dopak', 'mov'],
    'Diode_SMD': ['d', 'diode', 'diodes', 'zener', 'schottky', 'tvs'],
    'Connector': ['conn', 'connector', 'connectors', 'header', 'plug', 'jack', 'usb', 'terminal'],
    'Inductor_SMD': ['l', 'ind', 'inductor', 'choke', 'coil'],
    'Button_Switch_SMD': ['sw', 'switch', 'button', 'btn', 'tactile', 'toggle'],
    'LED_SMD': ['led', 'leds', 'light_emitting_diode'],
    'Transformer_SMD': ['t', 'transformer', 'xfmr'],
    'PCB_Defect': ['defect', 'open', 'short', 'mousebite', 'spur', 'pin-hole', 'spurious_copper']
}

class LabelHomogenizer:
    def __init__(self):
        self.lookup = {}
        for std_name, synonyms in LABEL_SYNONYMS.items():
            for syn in synonyms:
                self.lookup[syn.lower()] = std_name
                
    def homogenize(self, raw_label):
        if isinstance(raw_label, int):
            return KICAD_CLASSES[raw_label] if raw_label < len(KICAD_CLASSES) else f"Class_{raw_label}"
        cleaned = str(raw_label).strip().lower()
        if cleaned in self.lookup:
            return self.lookup[cleaned]
        for syn, std_name in self.lookup.items():
            if syn in cleaned:
                return std_name
        return f"Unknown_{raw_label}"

homogenizer = LabelHomogenizer()

In [ ]:
# Step 3: LabelMe JSON Exporter Function
def export_to_labelme_json(image_name, img_w, img_h, polygon_records, output_json_path):
    """
    Exports SAM polygon points to standard LabelMe JSON format for interactive visual GUI editing.
    """
    shapes = []
    for rec in polygon_records:
        raw_lbl = rec.get('raw_label', rec.get('class_id'))
        kicad_name = homogenizer.homogenize(raw_lbl)
        
        # Convert normalized [0, 1] points to absolute pixel coordinates [x_px, y_px]
        pts_norm = rec['points']
        pixel_pts = []
        for i in range(0, len(pts_norm), 2):
            px = round(pts_norm[i] * img_w, 2)
            py = round(pts_norm[i+1] * img_h, 2)
            pixel_pts.append([px, py])
            
        shapes.append({
            "label": kicad_name,
            "points": pixel_pts,
            "group_id": None,
            "description": "SAM Auto-Annotation Overlay",
            "shape_type": "polygon",
            "flags": {}
        })
        
    labelme_data = {
        "version": "5.0.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": image_name,
        "imageData": None,
        "imageHeight": img_h,
        "imageWidth": img_w
    }
    
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(labelme_data, f, indent=2)
        
    return output_json_path

In [ ]:
# Demonstration of LabelMe JSON Output Format
sample_polygons = [
    {'raw_label': 'cap1', 'points': [0.1, 0.1, 0.2, 0.1, 0.2, 0.2, 0.1, 0.2]},
    {'raw_label': 'MOSFET', 'points': [0.4, 0.4, 0.6, 0.4, 0.6, 0.6, 0.4, 0.6]}
]

json_out = export_to_labelme_json("pcb_demo.jpg", 640, 640, sample_polygons, "pcb_demo.json")
print(f"Exported sample LabelMe JSON overlay: {json_out}")
with open(json_out, 'r') as f:
    print(f.read()[:500], "\n...")